# CausalChange quickstart



In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from pprint import pprint

import networkx as nx
import pandas as pd

from benchmarks.run_methods import (
    run_algo,
    run_sampling,
    run_scoring,
)
from causalchange.causal_change import CausalChange
from causalchange.config.benchmark_config import (
    BenchmarkConfig,
    MultiDataConfig,
    SingleDataConfig,
)
from causalchange.config.cc_types import (
    ContextMode,
    DataMode,
    GraphSearch,
    ScoreType,
)


def print_metrics(metrics):
    for key in sorted(metrics):
        print(f"{key:32s} {metrics[key]}")


def draw_graph(graph, title=None):
    if title:
        print(title)
    nx.draw(graph, with_labels=True)

## 1. IID synthetic data with TOPIC

This uses the benchmark generator only for convenience. The estimator itself is just `CausalChange(...).fit(df)`.


In [2]:
cfg_data = SingleDataConfig.model_validate(
    {
        "setting": "single",
        "n_nodes": 5,
        "edge_prob": 0.4,
        "n_samples": 1000,
        "nonlinearity": "tanh",
        "seed": 1,
    }
)

sample = run_sampling(cfg_data)
df = sample.df
true_g = sample.true_summary_dag

est = CausalChange(
    data_mode=DataMode.IID,
    graph_search=GraphSearch.TOPIC,
    score_type=ScoreType.GAM,
)

est.fit(df)

print("Data shape:", df.shape)
print("Estimated edges:")
pprint(sorted(est.graph.edges()))
print("\nTrue synthetic edges:")
pprint(sorted(true_g.edges()))

Data shape: (1000, 5)
Estimated edges:
[('X0', 'X4'), ('X2', 'X0'), ('X4', 'X3')]

True synthetic edges:
[('X0', 'X2'), ('X4', 'X0'), ('X4', 'X3')]


## 2. Multi-context synthetic data with LINC

The multi-context dataframe contains a context column. The context column name must be passed to `CausalChange`.


In [3]:
cfg_data = MultiDataConfig.model_validate(
    {
        "setting": "multi",
        "n_nodes": 5,
        "edge_prob": 0.4,
        "n_contexts": 4,
        "n_samples_per_context": 300,
        "n_intervened_per_context": 2,
        "context_col": "context",
        "intervention_type": "soft_weight",
        "nonlinearity": "tanh",
        "seed": 1,
    }
)

sample = run_sampling(cfg_data)
df = sample.df
true_g = sample.true_summary_dag

est = CausalChange(
    data_mode=DataMode.CONTEXTS,
    graph_search=GraphSearch.TOPIC,
    score_type=ScoreType.GAM,
    context_mode=ContextMode.LINC,
    context_col="context",
)

est.fit(df)

print("Data shape:", df.shape)
print("Context counts:")
print(df["context"].value_counts().sort_index())
print("\nEstimated edges:")
pprint(sorted(est.graph.edges()))
print("\nTrue synthetic edges:")
pprint(sorted(true_g.edges()))

Data shape: (1200, 6)
Context counts:
context
0    300
1    300
2    300
3    300
Name: count, dtype: int64

Estimated edges:
[('X2', 'X0'), ('X2', 'X4'), ('X4', 'X0'), ('X4', 'X3')]

True synthetic edges:
[('X0', 'X2'), ('X4', 'X0'), ('X4', 'X3')]




Set `csv_path` to a local CSV. For context data, set `context_col`. For IID data, leave `context_col = None`.

The feature columns should be numeric. If a context column is present, it is excluded from the causal variables.


In [4]:
csv_path = Path("path/to/your_data.csv")
context_col = None  # e.g. "context" for multi-context data

if csv_path.exists():
    df_user = pd.read_csv(csv_path)

    if context_col is None:
        data_mode = DataMode.IID
        aggregation = ContextMode.SKIP
    else:
        data_mode = DataMode.CONTEXTS
        aggregation = ContextMode.LINC

    est_user = CausalChange(
        data_mode=data_mode,
        graph_search=GraphSearch.TOPIC,
        score_type=ScoreType.GAM,
        context_mode=aggregation,
        context_col=context_col,
    )

    est_user.fit(df_user)

    print("Loaded:", csv_path)
    print("Data shape:", df_user.shape)
    print("Estimated edges:")
    pprint(sorted(est_user.graph.edges()))
else:
    print(f"Edit csv_path first. File does not exist: {csv_path}")

Edit csv_path first. File does not exist: path/to/your_data.csv


## 4. SpaceTime: graph discovery with oracle changepoints

This uses the new SpaceTime synthetic generator and benchmark wrapper.


In [5]:
cfg = BenchmarkConfig.model_validate(
    {
        "data": {
            "setting": "time",
            "n_nodes": 3,
            "edge_prob": 0.4,
            "n_samples": 120,
            "tau_max": 2,
            "seed": 1,
            "n_changepoints": 2,
            "n_regimes": 2,
            "min_segment_length": 30,
            "nonlinearity": "tanh",
        },
        "algo": {
            "name": "spacetime",
            "score_type": "ff",
            "changepoint_mode": "detect",
            "detect_contexts": False,
            "detect_regimes": False,
        },
    }
)

sample = run_sampling(cfg.data)
est = run_algo(sample, cfg.data, cfg.algo)
metrics, est_graph = run_scoring(sample, est, return_nx=True)

print("True changepoints:", sample.spacetime.changepoints)
print("Estimated changepoints:", est.result.changepoints)
print()
print_metrics(metrics)

True changepoints: [41, 79]
Estimated changepoints: [90]

changepoint_f1                   0.0
changepoint_mean_abs_error       nan
changepoint_n_est                1.0
changepoint_n_true               2.0
changepoint_precision            0.0
changepoint_recall               0.0
edge_f1                          0.5714285714285715
edge_precision                   1.0
edge_recall                      0.4
shd                              1.0
skel_f1                          0.8
skel_precision                   1.0
skel_recall                      0.6666666666666666
summary_edge_f1                  0.5714285714285715
summary_edge_precision           1.0
summary_edge_recall              0.4
summary_shd                      1.0
summary_skel_f1                  0.8
summary_skel_precision           1.0
summary_skel_recall              0.6666666666666666
wcg_edge_f1                      0.5333333333333333
wcg_edge_precision               0.8
wcg_edge_recall                  0.4
wcg_shd         